# ML Baseline: HARGPT Models with RAG-HAR Preprocessing

This notebook rebuilds the classical ML baseline for `outputs/hargpt_windows_0_to_352_65_GH040226/manual_labels/manual_labels_65_GH040226.csv`.

Model choices follow **HARGPT**:

- `RandomForestClassifier()` with default scikit-learn settings
- `SVC(kernel="rbf")`

Input representation follows **RAG-HAR-style preprocessing** because the HARGPT paper does not specify the classical-ML feature construction in enough detail to reproduce directly.

This notebook uses:

- `manual_label` as the single target label for each window
- Z-score normalization per channel
- sliding windows
- four segments per window: `full`, `start`, `mid`, `end`
- eight handcrafted statistics per segment/channel: `mean`, `max`, `min`, `q1`, `q3`, `std`, `median`, `peaks`
- flattened standardized raw `x/y/z` samples from each window

By default, this notebook runs the **full-timeline** task: `Still` is kept as the background class, so train/eval both include genuine background windows. Set `DROP_STILL_WINDOWS = True` only for an active-only ablation where background windows are excluded before splitting.


In [27]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from scripts.covfee_preprocessing import preprocess_covfee_annotations


In [28]:
COVFEE_OUTPUT_DIR = Path("outputs") / "covfee"
DATA_PATH = COVFEE_OUTPUT_DIR / "covfee_manual_labels_combined.csv"
SUMMARY_PATH = COVFEE_OUTPUT_DIR / "covfee_preprocessing_summary.csv"

# COVFEE pilot: preprocess the 10 manual-label JSON files first, then train the ML baseline
# from the generated combined CSV instead of reading a precomputed single-participant CSV.
combined_df, preprocessing_summary = preprocess_covfee_annotations(
    Path.cwd(),
    output_dir=COVFEE_OUTPUT_DIR,
)
if combined_df.empty:
    raise ValueError("COVFEE preprocessing did not produce any labeled ACC rows.")

CHANNEL_COLUMNS = ["x", "y", "z"]
TARGET_COLUMN = "manual_label"
GROUP_COLUMNS = ["annotation_id"]
WINDOW_SECONDS = 1.0
STRIDE_SECONDS = 1.0
FEATURE_MODE = "stats_plus_raw"  # options: stats_only, raw_flattened, stats_plus_raw
TARGET_FS = None  # set to 10 to match the HARGPT LLM token rate more closely
MIN_LABEL_PURITY = 0.0  # e.g. 0.9 to keep only nearly pure windows
BACKGROUND_LABEL = "Still"
DROP_STILL_WINDOWS = False  # False = full timeline; True = active-only ablation
CANONICAL_LABEL_ORDER = ["Still", "Gesture", "Nodding", "Drinking", "Toasting"]

SPLIT_CONFIGS = {
    "split_A_eval_01_v1": {
        "eval_annotations": ["01_v1", "20_v2", "27_v2"],
    },
    "split_B_eval_24_v1": {
        "eval_annotations": ["24_v1", "18_v2", "27_v2"],
    },
}
RANDOM_SEED = 42

DATA_PATH


WindowsPath('outputs/covfee/covfee_manual_labels_combined.csv')

In [29]:
df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values(GROUP_COLUMNS + ["time"]).reset_index(drop=True)

sampling_rows = []
for annotation_id, group_df in df.groupby("annotation_id", sort=False):
    positive_diffs = group_df["time"].diff().dt.total_seconds().dropna()
    positive_diffs = positive_diffs[positive_diffs > 0]
    if positive_diffs.empty:
        continue
    sampling_rows.append(
        {
            "annotation_id": annotation_id,
            "estimated_fs": 1.0 / positive_diffs.median(),
            "rows": len(group_df),
        }
    )
sampling_summary = pd.DataFrame(sampling_rows)
original_fs = float(sampling_summary["estimated_fs"].median())

print(f"Rows: {len(df):,}")
print(f"Annotations: {df['annotation_id'].nunique()}")
print(f"Median estimated sampling rate: {original_fs:.2f} Hz")
display(preprocessing_summary)
display(sampling_summary)
display(df.head())
display(df[TARGET_COLUMN].value_counts().rename("count").to_frame())


Rows: 8,228
Annotations: 1
Median estimated sampling rate: 50.00 Hz


,annotation_id,participant_no,participant_id,video_name,status,alignment_mode,rows,windows,video_start,video_end,...,acc_start,acc_end,overlap_seconds,label_file,acc_dir,acc_files,video_file,labeled_csv,window_index_csv,overlay_png
0,01_v1,1,81,v1,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 14:02:30.166833333,2025-07-17 14:07:30.166833333,...,2025-07-17 12:14:03.100,2025-07-17 16:32:18.460,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv;ACC_1.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\01_v1\manual_labels_01_v1.csv,outputs\covfee\01_v1\window_index_01_v1.csv,outputs\covfee\01_v1\manual_labels_overlay_01_...
1,03_v1,3,72,v1,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 14:02:30.166833333,2025-07-17 14:07:30.166833333,...,2025-07-17 12:12:31.320,2025-07-17 16:30:38.020,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv;ACC_1.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\03_v1\manual_labels_03_v1.csv,outputs\covfee\03_v1\window_index_03_v1.csv,outputs\covfee\03_v1\manual_labels_overlay_03_...
2,03_v2,3,72,v2,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 15:20:32.116783333,2025-07-17 15:25:32.116783333,...,2025-07-17 12:12:31.320,2025-07-17 16:30:38.020,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv;ACC_1.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\03_v2\manual_labels_03_v2.csv,outputs\covfee\03_v2\window_index_03_v2.csv,outputs\covfee\03_v2\manual_labels_overlay_03_...
3,16_v1,16,68,v1,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 14:02:30.166833333,2025-07-17 14:07:30.166833333,...,2025-07-17 12:12:02.640,2025-07-17 16:30:15.480,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\16_v1\manual_labels_16_v1.csv,outputs\covfee\16_v1\window_index_16_v1.csv,outputs\covfee\16_v1\manual_labels_overlay_16_...
4,16_v2,16,68,v2,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 15:20:32.116783333,2025-07-17 15:25:32.116783333,...,2025-07-17 12:12:02.640,2025-07-17 16:30:15.480,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\16_v2\manual_labels_16_v2.csv,outputs\covfee\16_v2\window_index_16_v2.csv,outputs\covfee\16_v2\manual_labels_overlay_16_...
5,17_v1,17,77,v1,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 14:02:30.166833333,2025-07-17 14:07:30.166833333,...,2025-07-17 12:13:22.240,2025-07-17 16:31:50.340,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\17_v1\manual_labels_17_v1.csv,outputs\covfee\17_v1\window_index_17_v1.csv,outputs\covfee\17_v1\manual_labels_overlay_17_...
6,18_v2,18,89,v2,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 15:20:32.116783333,2025-07-17 15:25:32.116783333,...,2025-07-17 12:17:47.860,2025-07-17 16:33:15.960,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\18_v2\manual_labels_18_v2.csv,outputs\covfee\18_v2\window_index_18_v2.csv,outputs\covfee\18_v2\manual_labels_overlay_18_...
7,20_v2,20,93,v2,ok,hardcoded_source_clip_offset,15000,150,2025-07-17 15:20:32.116783333,2025-07-17 15:25:32.116783333,...,2025-07-17 12:17:09.180,2025-07-17 16:33:42.780,300.0,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,ACC_0.csv,G:\TUD_CESE\OneDrive - Delft University of Tec...,outputs\covfee\20_v2\manual_labels_20_v2.csv,outputs\covfee\20_v2\window

,annotation_id,estimated_fs,rows
0,01_v1,50.0,8228


,time,x,y,z,state,manual_label,video_name,participant_no,participant_id,annotation_id,alignment_mode,preprocessed_at
0,2025-07-17 14:02:30.180,-0.197876,-0.151733,-0.875854,0,Still,v1,1,81,01_v1,hardcoded_source_clip_offset,2026-05-05 11:58:47
1,2025-07-17 14:02:30.200,-0.194946,-0.140503,-0.872925,0,Still,v1,1,81,01_v1,hardcoded_source_clip_offset,2026-05-05 11:58:47
2,2025-07-17 14:02:30.220,-0.158813,-0.128296,-0.875366,0,Still,v1,1,81,01_v1,hardcoded_source_clip_offset,2026-05-05 11:58:47
3,2025-07-17 14:02:30.240,-0.170044,-0.129272,-0.915894,0,Still,v1,1,81,01_v1,hardcoded_source_clip_offset,2026-05-05 11:58:47
4,2025-07-17 14:02:30.260,-0.178345,-0.126831,-0.951050,0,Still,v1,1,81,01_v1,hardcoded_source_clip_offset,2026-05-05 11:58:47


,count
manual_label,
Still,3721
Gesture,3193
Nodding,1276
Toasting,38


In [30]:
def maybe_downsample(frame: pd.DataFrame, original_fs: float, target_fs: float | None) -> pd.DataFrame:
    if target_fs is None or target_fs >= original_fs:
        return frame.copy()

    step = max(1, int(round(original_fs / target_fs)))
    return frame.iloc[::step].reset_index(drop=True)


def zscore_channels(frame: pd.DataFrame, channel_columns: list[str]) -> pd.DataFrame:
    out = frame.copy()
    for col in channel_columns:
        mean = out[col].mean()
        std = out[col].std()
        if pd.isna(std) or std == 0:
            std = 1.0
        out[col] = (out[col] - mean) / std
    return out


def count_peaks(values: np.ndarray) -> int:
    if len(values) < 3:
        return 0
    left = values[1:-1] > values[:-2]
    right = values[1:-1] > values[2:]
    return int(np.sum(left & right))


def extract_segment_features(values: np.ndarray, prefix: str) -> dict:
    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_max": float(np.max(values)),
        f"{prefix}_min": float(np.min(values)),
        f"{prefix}_q1": float(np.quantile(values, 0.25)),
        f"{prefix}_q3": float(np.quantile(values, 0.75)),
        f"{prefix}_std": float(np.std(values)),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_peaks": count_peaks(values),
    }


def extract_raw_flattened_features(window: pd.DataFrame, channel_columns: list[str]) -> dict:
    features = {}
    for channel in channel_columns:
        values = window[channel].to_numpy(dtype=np.float64)
        for idx, value in enumerate(values):
            features[f"{channel}_raw_{idx:03d}"] = float(value)
    return features


def extract_window_features(window: pd.DataFrame, channel_columns: list[str]) -> tuple[dict, float, int]:
    features = {}
    label_distribution = window[TARGET_COLUMN].value_counts(normalize=True)
    purity = float(label_distribution.iloc[0])
    n_unique_labels = int(label_distribution.shape[0])

    partitions = {
        "full": window,
        "start": window.iloc[: len(window) // 3],
        "mid": window.iloc[len(window) // 3 : (2 * len(window)) // 3],
        "end": window.iloc[(2 * len(window)) // 3 :],
    }

    if FEATURE_MODE in {"stats_only", "stats_plus_raw"}:
        for channel in channel_columns:
            for segment_name, segment_df in partitions.items():
                values = segment_df[channel].to_numpy(dtype=np.float64)
                features.update(extract_segment_features(values, f"{channel}_{segment_name}"))

    if FEATURE_MODE in {"raw_flattened", "stats_plus_raw"}:
        features.update(extract_raw_flattened_features(window, channel_columns))

    return features, purity, n_unique_labels


def build_feature_table(frame: pd.DataFrame, sampling_rate: float, window_seconds: float, stride_seconds: float, channel_columns: list[str]) -> tuple[pd.DataFrame, int, int]:
    window_size = int(round(window_seconds * sampling_rate))
    stride_size = int(round(stride_seconds * sampling_rate))
    rows = []

    for start in range(0, len(frame) - window_size + 1, stride_size):
        end = start + window_size
        window = frame.iloc[start:end]
        feature_dict, purity, n_unique_labels = extract_window_features(window, channel_columns)
        row = {
            "start_idx": start,
            "end_idx": end,
            "start_time": window["time"].iloc[0],
            "end_time": window["time"].iloc[-1],
            "purity": purity,
            "n_unique_labels": n_unique_labels,
            TARGET_COLUMN: window[TARGET_COLUMN].mode().iloc[0],
        }
        row.update(feature_dict)
        rows.append(row)

    return pd.DataFrame(rows), window_size, stride_size


def active_label_order(y_true: np.ndarray, y_pred: np.ndarray) -> list[str]:
    active_values = set(np.asarray(y_true)) | set(np.asarray(y_pred))
    return [
        label
        for label in CANONICAL_LABEL_ORDER
        if label != BACKGROUND_LABEL and label in active_values
    ]


def evaluate_split(name: str, y_true: pd.Series, y_pred: np.ndarray) -> dict:
    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_array,
        y_pred_array,
        average="macro",
        zero_division=0,
    )

    active_mask = y_true_array != BACKGROUND_LABEL
    active_accuracy = np.nan
    active_macro_precision = np.nan
    active_macro_recall = np.nan
    active_macro_f1 = np.nan
    active_n_samples = int(active_mask.sum())
    if active_n_samples > 0:
        active_accuracy = accuracy_score(y_true_array[active_mask], y_pred_array[active_mask])
        labels_without_still = active_label_order(y_true_array[active_mask], y_pred_array[active_mask])
        if labels_without_still:
            active_macro_precision, active_macro_recall, active_macro_f1, _ = precision_recall_fscore_support(
                y_true_array[active_mask],
                y_pred_array[active_mask],
                labels=labels_without_still,
                average="macro",
                zero_division=0,
            )

    return {
        "split": name,
        "accuracy": accuracy_score(y_true_array, y_pred_array),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "active_accuracy": active_accuracy,
        "active_macro_precision_no_still": active_macro_precision,
        "active_macro_recall_no_still": active_macro_recall,
        "active_macro_f1_no_still": active_macro_f1,
        "n_samples": len(y_true_array),
        "active_n_samples": active_n_samples,
    }


def evaluate_timeline_diagnostics(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)
    background_mask = y_true_array == BACKGROUND_LABEL
    active_mask = ~background_mask
    active_labels = active_label_order(y_true_array[active_mask], y_pred_array[active_mask])

    background_recall = np.nan
    false_activity_rate = np.nan
    if background_mask.any():
        background_recall = float(np.mean(y_pred_array[background_mask] == BACKGROUND_LABEL))
        false_activity_rate = 1.0 - background_recall

    active_accuracy = np.nan
    active_macro_f1 = np.nan
    if active_mask.any():
        active_accuracy = accuracy_score(y_true_array[active_mask], y_pred_array[active_mask])
        if active_labels:
            active_macro_f1 = precision_recall_fscore_support(
                y_true_array[active_mask],
                y_pred_array[active_mask],
                labels=active_labels,
                average="macro",
                zero_division=0,
            )[2]

    return {
        "split": name,
        "background_windows": int(background_mask.sum()),
        "active_windows": int(active_mask.sum()),
        "background_recall": background_recall,
        "false_activity_rate_on_background": false_activity_rate,
        "active_accuracy_on_active_windows": active_accuracy,
        "active_macro_f1_no_still_on_active_windows": active_macro_f1,
    }


In [31]:
feature_tables = []
window_size = None
stride_size = None

for annotation_id, group_df in df.groupby("annotation_id", sort=False):
    positive_diffs = group_df["time"].diff().dt.total_seconds().dropna()
    positive_diffs = positive_diffs[positive_diffs > 0]
    if positive_diffs.empty:
        continue
    group_original_fs = 1.0 / positive_diffs.median()
    model_group_df = maybe_downsample(group_df, original_fs=group_original_fs, target_fs=TARGET_FS)
    model_group_df = zscore_channels(model_group_df, CHANNEL_COLUMNS)
    group_effective_fs = group_original_fs if TARGET_FS is None else min(TARGET_FS, group_original_fs)

    group_feature_df, group_window_size, group_stride_size = build_feature_table(
        model_group_df,
        sampling_rate=group_effective_fs,
        window_seconds=WINDOW_SECONDS,
        stride_seconds=STRIDE_SECONDS,
        channel_columns=CHANNEL_COLUMNS,
    )
    if group_feature_df.empty:
        continue

    group_feature_df["annotation_id"] = annotation_id
    for col in ["participant_no", "participant_id", "video_name", "alignment_mode"]:
        if col in model_group_df.columns:
            group_feature_df[col] = model_group_df[col].iloc[0]
    group_feature_df["estimated_fs"] = group_effective_fs
    feature_tables.append(group_feature_df)
    window_size = group_window_size if window_size is None else window_size
    stride_size = group_stride_size if stride_size is None else stride_size

if not feature_tables:
    raise ValueError("No COVFEE windows could be built. Check preprocessing_summary for skipped annotations.")

feature_df = pd.concat(feature_tables, ignore_index=True)
n_windows_before_filter = len(feature_df)
purity_mask = feature_df["purity"] >= MIN_LABEL_PURITY
background_mask = feature_df[TARGET_COLUMN] == BACKGROUND_LABEL
selection_mask = purity_mask.copy()
if DROP_STILL_WINDOWS:
    selection_mask &= ~background_mask

n_removed_by_purity = int((~purity_mask).sum())
n_removed_background = int((purity_mask & background_mask).sum()) if DROP_STILL_WINDOWS else 0
feature_df = feature_df[selection_mask].reset_index(drop=True)
if feature_df.empty:
    raise ValueError("No windows left after purity/background filtering. Relax MIN_LABEL_PURITY or set DROP_STILL_WINDOWS=False.")

effective_fs = original_fs if TARGET_FS is None else min(TARGET_FS, original_fs)

print(f"Feature annotations: {feature_df['annotation_id'].nunique()}")
print(f"Median effective sampling rate: {effective_fs:.2f} Hz")
print(f"Window size: {window_size} samples")
print(f"Stride size: {stride_size} samples")
print(f"Feature mode: {FEATURE_MODE}")
print(f"Drop Still/background windows: {DROP_STILL_WINDOWS}")
print(f"Removed by purity filter: {n_removed_by_purity:,}")
print(f"Removed Still/background windows: {n_removed_background:,}")
print(f"Feature table shape: {feature_df.shape}")
display(feature_df[["annotation_id", "start_time", "end_time", TARGET_COLUMN, "purity", "n_unique_labels"]].head())


Feature annotations: 1
Median effective sampling rate: 50.00 Hz
Window size: 50 samples
Stride size: 50 samples
Feature mode: stats_plus_raw
Drop Still/background windows: False
Removed by purity filter: 0
Removed Still/background windows: 0
Feature table shape: (164, 259)


,annotation_id,start_time,end_time,manual_label,purity,n_unique_labels
0,01_v1,2025-07-17 14:02:30.180,2025-07-17 14:02:31.160,Gesture,0.7,2
1,01_v1,2025-07-17 14:02:31.180,2025-07-17 14:02:32.160,Gesture,1.0,1
2,01_v1,2025-07-17 14:02:32.180,2025-07-17 14:02:33.160,Gesture,1.0,1
3,01_v1,2025-07-17 14:02:33.180,2025-07-17 14:02:34.160,Gesture,1.0,1
4,01_v1,2025-07-17 14:02:34.180,2025-07-17 14:02:35.160,Gesture,1.0,1


## Manual-label Purity Check

This quantifies how much information is compressed when a mixed-label window is assigned a single `manual_label` by majority vote.


In [32]:
purity_summary = pd.DataFrame(
    {
        "metric": [
            "n_windows",
            "mean_purity",
            "median_purity",
            "pure_windows_ratio",
            "mixed_windows_ratio",
            "purity_ge_0.9_ratio",
            "purity_ge_0.8_ratio",
        ],
        "value": [
            len(feature_df),
            feature_df["purity"].mean(),
            feature_df["purity"].median(),
            (feature_df["purity"] == 1.0).mean(),
            (feature_df["purity"] < 1.0).mean(),
            (feature_df["purity"] >= 0.9).mean(),
            (feature_df["purity"] >= 0.8).mean(),
        ],
    }
)
display(purity_summary)
display(feature_df[["purity", "n_unique_labels"]].describe())


,metric,value
0,n_windows,164.000000
1,mean_purity,0.905366
2,median_purity,1.000000
3,pure_windows_ratio,0.670732
4,mixed_windows_ratio,0.329268
5,purity_ge_0.9_ratio,0.731707
6,purity_ge_0.8_ratio,0.792683


,purity,n_unique_labels
count,164.000000,164.000000
mean,0.905366,1.341463
std,0.164344,0.500785
min,0.380000,1.000000
25%,0.870000,1.000000
50%,1.000000,1.000000
75%,1.000000,2.000000
max,1.000000,3.000000


In [33]:
non_feature_columns = {
    "start_idx",
    "end_idx",
    "start_time",
    "end_time",
    "purity",
    "n_unique_labels",
    TARGET_COLUMN,
    "annotation_id",
    "participant_no",
    "participant_id",
    "video_name",
    "alignment_mode",
    "estimated_fs",
}
feature_columns = [c for c in feature_df.columns if c not in non_feature_columns]
label_order = [label for label in CANONICAL_LABEL_ORDER if label in feature_df[TARGET_COLUMN].unique()]
feature_df = feature_df.sort_values(["video_name", "participant_no", "start_time"]).reset_index(drop=True)

available_annotations = set(feature_df["annotation_id"].unique())
splits = {}
split_summary_rows = []
label_count_rows = []
annotation_split_rows = []

for split_name, split_config in SPLIT_CONFIGS.items():
    eval_annotations = set(split_config["eval_annotations"])
    missing_annotations = sorted(eval_annotations - available_annotations)
    if missing_annotations:
        raise ValueError(f"{split_name} references missing annotations: {missing_annotations}")

    eval_mask = feature_df["annotation_id"].isin(eval_annotations)
    train_df = feature_df[~eval_mask].reset_index(drop=True)
    eval_df = feature_df[eval_mask].reset_index(drop=True)
    if train_df.empty or eval_df.empty:
        raise ValueError(f"{split_name} produced an empty train or eval split.")

    splits[split_name] = {
        "train_df": train_df,
        "eval_df": eval_df,
        "X_train": train_df[feature_columns],
        "y_train": train_df[TARGET_COLUMN],
        "X_eval": eval_df[feature_columns],
        "y_eval": eval_df[TARGET_COLUMN],
    }

    for subset_name, subset_df in [("train", train_df), ("eval", eval_df)]:
        split_summary_rows.append(
            {
                "split_config": split_name,
                "subset": subset_name,
                "n_windows": len(subset_df),
                "annotations": subset_df["annotation_id"].nunique(),
                "start": subset_df["start_time"].min(),
                "end": subset_df["end_time"].max(),
            }
        )
        for label, count in subset_df[TARGET_COLUMN].value_counts().reindex(label_order, fill_value=0).items():
            label_count_rows.append(
                {
                    "split_config": split_name,
                    "subset": subset_name,
                    "label": label,
                    "n_windows": int(count),
                }
            )
        for annotation_id, count in subset_df["annotation_id"].value_counts().sort_index().items():
            annotation_split_rows.append(
                {
                    "split_config": split_name,
                    "subset": subset_name,
                    "annotation_id": annotation_id,
                    "n_windows": int(count),
                }
            )

split_summary = pd.DataFrame(split_summary_rows)
label_counts_df = pd.DataFrame(label_count_rows).pivot_table(
    index=["split_config", "label"],
    columns="subset",
    values="n_windows",
    fill_value=0,
).astype(int).reindex(label_order, level="label")
annotation_split_df = pd.DataFrame(annotation_split_rows).pivot_table(
    index=["split_config", "annotation_id"],
    columns="subset",
    values="n_windows",
    fill_value=0,
).astype(int)

display(split_summary)
display(label_counts_df)
display(annotation_split_df)


ValueError: split_A_eval_01_v1 references missing annotations: ['20_v2', '27_v2']

maybe remove still windows to focus on more dynamic activities?

experiments with levels of mdoel approach complexity:
svm/random_forest
cnn/lstm/transformer on raw windows
llm: prompt engineering with window-level features vs raw data, zero-shot vs few-shot, etc. preprocessing variations: window size/stride, feature engineering vs raw, purity thresholds, etc.

in the end: answer the question how usable is the data and which modeling approaches work best, to what extent, and why? (e.g. which activities are most/least confused and why? what are the main sources of error?)

In [ ]:
models = {
    "random_forest": RandomForestClassifier(random_state=RANDOM_SEED),
    "svm_rbf": make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=RANDOM_SEED)),
}
models


In [ ]:
train_results = []
eval_split_results = []
eval_annotation_results = []
reports = {}
predictions = {}
fitted_models = {}

eval_label_order = [label for label in CANONICAL_LABEL_ORDER if label in feature_df[TARGET_COLUMN].unique()]

for split_name, split_data in splits.items():
    reports[split_name] = {}
    predictions[split_name] = {}
    fitted_models[split_name] = {}

    X_train, y_train = split_data["X_train"], split_data["y_train"]
    X_eval, y_eval = split_data["X_eval"], split_data["y_eval"]
    eval_df = split_data["eval_df"]

    for model_name, model in models.items():
        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)
        fitted_models[split_name][model_name] = fitted_model
        predictions[split_name][model_name] = {
            "train": fitted_model.predict(X_train),
            "eval": fitted_model.predict(X_eval),
        }
        train_pred = predictions[split_name][model_name]["train"]
        eval_pred = predictions[split_name][model_name]["eval"]

        reports[split_name][model_name] = {
            "eval_split": classification_report(y_eval, eval_pred, labels=eval_label_order, zero_division=0),
            "eval_annotations": {},
        }

        train_results.append(
            {
                "split_config": split_name,
                "annotation_id": "ALL_TRAIN",
                "model": model_name,
                **evaluate_split("train", y_train, train_pred),
            }
        )
        eval_split_results.append(
            {
                "split_config": split_name,
                "annotation_id": "ALL_EVAL",
                "model": model_name,
                **evaluate_split("eval_split", y_eval, eval_pred),
            }
        )

        for annotation_id in eval_df["annotation_id"].drop_duplicates():
            annotation_mask = eval_df["annotation_id"].eq(annotation_id).to_numpy()
            y_eval_annotation = y_eval.to_numpy()[annotation_mask]
            eval_pred_annotation = eval_pred[annotation_mask]
            reports[split_name][model_name]["eval_annotations"][annotation_id] = classification_report(
                y_eval_annotation,
                eval_pred_annotation,
                labels=eval_label_order,
                zero_division=0,
            )
            eval_annotation_results.append(
                {
                    "split_config": split_name,
                    "annotation_id": annotation_id,
                    "model": model_name,
                    **evaluate_split("eval_annotation", y_eval_annotation, eval_pred_annotation),
                }
            )

results_df = pd.DataFrame(train_results + eval_annotation_results)
eval_split_summary_df = pd.DataFrame(eval_split_results)

timeline_diagnostics = []
eval_split_timeline_diagnostics = []
for split_name, split_data in splits.items():
    y_train = split_data["y_train"]
    y_eval = split_data["y_eval"]
    eval_df = split_data["eval_df"]
    for model_name in models:
        train_pred = predictions[split_name][model_name]["train"]
        eval_pred = predictions[split_name][model_name]["eval"]
        timeline_diagnostics.append(
            {
                "split_config": split_name,
                "annotation_id": "ALL_TRAIN",
                "model": model_name,
                **evaluate_timeline_diagnostics("train", y_train, train_pred),
            }
        )
        eval_split_timeline_diagnostics.append(
            {
                "split_config": split_name,
                "annotation_id": "ALL_EVAL",
                "model": model_name,
                **evaluate_timeline_diagnostics("eval_split", y_eval, eval_pred),
            }
        )
        for annotation_id in eval_df["annotation_id"].drop_duplicates():
            annotation_mask = eval_df["annotation_id"].eq(annotation_id).to_numpy()
            timeline_diagnostics.append(
                {
                    "split_config": split_name,
                    "annotation_id": annotation_id,
                    "model": model_name,
                    **evaluate_timeline_diagnostics(
                        "eval_annotation",
                        y_eval.to_numpy()[annotation_mask],
                        eval_pred[annotation_mask],
                    ),
                }
            )

timeline_diagnostics_df = pd.DataFrame(timeline_diagnostics)
eval_split_timeline_diagnostics_df = pd.DataFrame(eval_split_timeline_diagnostics)

display(results_df)
display(eval_split_summary_df)
display(timeline_diagnostics_df)
display(eval_split_timeline_diagnostics_df)


In [ ]:
for split_name, split_reports in reports.items():
    for model_name, model_reports in split_reports.items():
        for annotation_id, report_text in model_reports["eval_annotations"].items():
            print(f"=== {split_name} / {annotation_id} / {model_name} / eval_annotation ===")
            print(report_text)


In [ ]:
LABEL_COLORS = {
    "Still": "#d9d9d9",
    "Gesture": "#f4a261",
    "Drinking": "#2a9d8f",
    "Toasting": "#577590",
    "Nodding": "#b56576",
}


def build_sample_runs(frame: pd.DataFrame, time_col: str, label_col: str, origin: pd.Timestamp):
    frame = frame[[time_col, label_col]].sort_values(time_col).reset_index(drop=True)
    runs = []
    if frame.empty:
        return runs
    start_idx = 0
    for i in range(1, len(frame) + 1):
        if i == len(frame) or frame.loc[i, label_col] != frame.loc[i - 1, label_col]:
            runs.append(
                (
                    (frame.loc[start_idx, time_col] - origin).total_seconds(),
                    (frame.loc[i - 1, time_col] - origin).total_seconds(),
                    frame.loc[i - 1, label_col],
                )
            )
            start_idx = i
    return runs


def build_interval_runs(frame: pd.DataFrame, start_col: str, end_col: str, label_col: str, origin: pd.Timestamp):
    frame = frame[[start_col, end_col, label_col]].sort_values(start_col).reset_index(drop=True)
    runs = []
    if frame.empty:
        return runs

    gap_tolerance = pd.Timedelta(milliseconds=25)
    current_start = frame.loc[0, start_col]
    current_end = frame.loc[0, end_col]
    current_label = frame.loc[0, label_col]

    for i in range(1, len(frame)):
        row = frame.loc[i]
        if row[label_col] == current_label and row[start_col] <= current_end + gap_tolerance:
            current_end = max(current_end, row[end_col])
        else:
            runs.append(
                (
                    (current_start - origin).total_seconds(),
                    (current_end - origin).total_seconds(),
                    current_label,
                )
            )
            current_start = row[start_col]
            current_end = row[end_col]
            current_label = row[label_col]

    runs.append(
        (
            (current_start - origin).total_seconds(),
            (current_end - origin).total_seconds(),
            current_label,
        )
    )
    return runs


for split_name, split_data in splits.items():
    for model_name in models:
        eval_with_pred = split_data["eval_df"].assign(predicted_label=predictions[split_name][model_name]["eval"])
        for plot_annotation_id in eval_with_pred["annotation_id"].drop_duplicates():
            eval_plot_df = eval_with_pred[eval_with_pred["annotation_id"] == plot_annotation_id].reset_index(drop=True)

            plot_start = eval_plot_df["start_time"].iloc[0]
            plot_end = eval_plot_df["end_time"].iloc[-1]
            plot_origin = plot_start

            acc_plot = df[
                (df["annotation_id"] == plot_annotation_id)
                & (df["time"] >= plot_start)
                & (df["time"] <= plot_end)
            ].copy().reset_index(drop=True)
            acc_plot["time_rel_s"] = (acc_plot["time"] - plot_origin).dt.total_seconds()

            manual_runs = build_sample_runs(acc_plot, "time", TARGET_COLUMN, plot_origin)
            predicted_runs = build_interval_runs(eval_plot_df, "start_time", "end_time", "predicted_label", plot_origin)

            fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
            ax0, ax1 = axes

            for start_time, end_time, predicted_label in predicted_runs:
                ax0.axvspan(start_time, end_time, color=LABEL_COLORS.get(str(predicted_label), "#cccccc"), alpha=0.18)

            ax0.plot(acc_plot["time_rel_s"], acc_plot["x"], label="x", color="blue", linewidth=0.7)
            ax0.plot(acc_plot["time_rel_s"], acc_plot["y"], label="y", color="red", linewidth=0.7)
            ax0.plot(acc_plot["time_rel_s"], acc_plot["z"], label="z", color="black", linewidth=0.7)
            ax0.set_title(f"Eval Timeline Comparison: {split_name} / {model_name} / {plot_annotation_id}")
            ax0.set_ylabel("Acceleration")
            ax0.legend(loc="upper right")

            for start_time, end_time, predicted_label in predicted_runs:
                ax1.axvspan(start_time, end_time, ymin=0.52, ymax=0.98, color=LABEL_COLORS.get(str(predicted_label), "#cccccc"), alpha=0.45)
            for start_time, end_time, manual_label in manual_runs:
                ax1.axvspan(start_time, end_time, ymin=0.02, ymax=0.48, color=LABEL_COLORS.get(str(manual_label), "#cccccc"), alpha=0.45)

            ax1.set_ylim(0, 1)
            ax1.set_yticks([0.25, 0.75])
            ax1.set_yticklabels(["Manual label", "Prediction"])
            ax1.set_title("Manual labels vs predictions")
            ax1.set_xlabel("Relative time within plotted eval segment (s)")

            legend_handles = [Patch(facecolor=color, edgecolor="gray", label=label) for label, color in LABEL_COLORS.items() if label in label_order]
            ax1.legend(handles=legend_handles, loc="upper right", ncol=min(len(legend_handles), 5))

            fig.tight_layout()
            plt.show()


In [ ]:
n_rows = len(splits) * len(models) * 3
fig, axes = plt.subplots(n_rows, 1, figsize=(6, 4 * n_rows))
if n_rows == 1:
    axes = np.array([axes])
else:
    axes = np.asarray(axes).reshape(-1)

row_idx = 0
for split_name, split_data in splits.items():
    y_eval = split_data["y_eval"]
    eval_df = split_data["eval_df"]
    for model_name in models:
        eval_pred = predictions[split_name][model_name]["eval"]
        for annotation_id in eval_df["annotation_id"].drop_duplicates():
            annotation_mask = eval_df["annotation_id"].eq(annotation_id).to_numpy()
            ConfusionMatrixDisplay.from_predictions(
                y_eval.to_numpy()[annotation_mask],
                eval_pred[annotation_mask],
                labels=label_order,
                display_labels=label_order,
                xticks_rotation=45,
                colorbar=False,
                cmap="Blues",
                ax=axes[row_idx],
            )
            axes[row_idx].set_title(f"{split_name} / {annotation_id} / {model_name} / eval_annotation")
            row_idx += 1

fig.tight_layout()
plt.show()
